In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
#from pmdarima.arima import auto_arima
from math import sqrt
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.arima_model import ARIMA
from statsmodels.tsa.stattools import adfuller
import statsmodels.tsa.stattools as tsa
from numpy import log

In [ ]:
# ORTHO-EAST

df_desc = pd.read_csv("data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv")
df_asc = pd.read_csv("data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv")

In [ ]:
df_desc

In [ ]:
df_asc

In [ ]:
# Pega os nomes das primeiras 25 colunas
colunas_desc = df_desc.columns[:24]

# Mostra os nomes
print(colunas_desc)

In [ ]:
# Pega os nomes das primeiras 25 colunas
colunas_asc = df_asc.columns[:24]

# Mostra os nomes
print(colunas_asc)

In [ ]:
# Definir os limites Alqueva
norte_min = 1855050
norte_max = 1855850
este_min = 2792250
este_max = 2793250

# Aplicar o filtro
df_desc = df_desc[
    (df_desc['northing'] >= norte_min) & (df_desc['northing'] <= norte_max) &
    (df_desc['easting'] >= este_min) & (df_desc['easting'] <= este_max)
]

df_desc

In [ ]:
# Definir os limites Alqueva
norte_min = 1855050
norte_max = 1855850
este_min = 2792250
este_max = 2793250

# Aplicar o filtro
df_asc = df_asc[
    (df_asc['northing'] >= norte_min) & (df_asc['northing'] <= norte_max) &
    (df_asc['easting'] >= este_min) & (df_asc['easting'] <= este_max)
]

df_asc

In [ ]:
import matplotlib.pyplot as plt

# Suponha que teu dataset seja um DataFrame chamado df
# E a variável que queres plotar é 'minha_variavel'

plt.figure(figsize=(8, 3))
plt.hist(df_asc['temporal_coherence'], bins=30, color='skyblue', edgecolor='black')
plt.title('Histograma da Variável')
plt.xlabel('Valor')
plt.ylabel('Frequência')
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Suponha que teu dataset seja um DataFrame chamado df
# E a variável que queres plotar é 'minha_variavel'

plt.figure(figsize=(8, 3))
plt.hist(df_desc['temporal_coherence'], bins=30, color='skyblue', edgecolor='black')
plt.title('Histograma da Variável')
plt.xlabel('Valor')
plt.ylabel('Frequência')
plt.grid(True)
plt.show()

# 🎯 Objetivo Final
- Transformar os deslocamentos LOS (line-of-sight) dos datasets ascending e descending para obter:

- Deslocamento Leste-Oeste (East)

- eslocamento Norte-Sul (North)

- Deslocamento Vertical (Up/Down)

- E depois fundir os dois conjuntos mesmo que os pontos não coincidam, para ter um campo vetorial completo.

# 📦 Datasets: df_asc e df_desc

## Cada dataset tem:

- 25 colunas iniciais fixas com metadados

- A partir da coluna 26, datas como colunas, com os deslocamentos LOS

- Colunas importantes: 'los_east', 'los_north', 'los_up' → componentes unitários do vetor LOS

# ✅ Etapa 1: Decompor deslocamentos LOS em componentes ENU (East, North, Up)

## Função

In [ ]:
import pandas as pd

def calcular_componentes_enu(displacements, los_vectors):
    """
    Transforma deslocamentos LOS em componentes East, North, Up
    - displacements: DataFrame com deslocamentos LOS (colunas = datas)
    - los_vectors: ndarray (n x 3) com [los_east, los_north, los_up]
    Retorna: 3 DataFrames: east, north, up
    """
    east = pd.DataFrame(index=displacements.index)
    north = pd.DataFrame(index=displacements.index)
    up = pd.DataFrame(index=displacements.index)

    for date in displacements.columns:
        d_los = displacements[date].values
        east[date] = d_los * los_vectors[:, 0]
        north[date] = d_los * los_vectors[:, 1]
        up[date] = d_los * los_vectors[:, 2]

    return east, north, up


## Aplicação

In [ ]:
# Separar deslocamentos
n_colunas_fixas = 25
asc_disp = df_asc.iloc[:, n_colunas_fixas:]
desc_disp = df_desc.iloc[:, n_colunas_fixas:]

# Extrair vetores LOS
asc_los = df_asc[['los_east', 'los_north', 'los_up']].values
desc_los = df_desc[['los_east', 'los_north', 'los_up']].values

# Converter para ENU
asc_east, asc_north, asc_up = calcular_componentes_enu(asc_disp, asc_los)
desc_east, desc_north, desc_up = calcular_componentes_enu(desc_disp, desc_los)


# 🚧 Etapa 2: Fusão dos dois datasets

## 🔁 Opção A — Interpolação Espacial

### Interpolar os pontos desc para a localização dos pontos asc e fazer a média:

In [ ]:
from scipy.interpolate import griddata
import numpy as np

def interpolar_componentes(desc_component, df_desc, df_asc):
    """
    Interpola um componente do df_desc para os pontos do df_asc
    """
    pontos_desc = df_desc[['easting', 'northing']].values
    pontos_asc = df_asc[['easting', 'northing']].values

    interpolado = pd.DataFrame(index=df_asc.index)

    for date in desc_component.columns:
        valores = desc_component[date].values
        interp = griddata(pontos_desc, valores, pontos_asc, method='linear')
        interpolado[date] = interp

    return interpolado


## Aplicação

In [ ]:
# Interpolação
desc_east_interp = interpolar_componentes(desc_east, df_desc, df_asc)
desc_north_interp = interpolar_componentes(desc_north, df_desc, df_asc)
desc_up_interp = interpolar_componentes(desc_up, df_desc, df_asc)

# Fusão (média simples dos dois)
east_fusion = (asc_east + desc_east_interp) / 2
north_fusion = (asc_north + desc_north_interp) / 2
up_fusion = (asc_up + desc_up_interp) / 2


In [ ]:
east_fusion

# 📊 Etapa 3: Visualização simples

## Exemplo 1 — Gráfico de linha dos deslocamentos médios:

In [ ]:
import matplotlib.pyplot as plt

# Deslocamento médio no tempo (só para exemplo)
east_mean = east_fusion.mean()
north_mean = north_fusion.mean()
up_mean = up_fusion.mean()

plt.figure(figsize=(12, 5))
plt.plot(east_mean, label='Leste', color='blue')
plt.plot(north_mean, label='Norte', color='green')
plt.plot(up_mean, label='Vertical', color='red')
plt.legend()
plt.title("Média dos deslocamentos por data")
plt.xlabel("Data")
plt.ylabel("Deslocamento (m)")
plt.grid()
plt.tight_layout()
plt.show()


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
import contextily as ctx

# === CONFIGURAÇÃO ===
# Componente a visualizar (ex: up_fusion, east_fusion, north_fusion)
componente = up_fusion  # troca aqui se quiseres outro
nome_componente = "up_fusion"

# Adiciona média da série temporal ao DataFrame original
df_asc[f"mean_{nome_componente}"] = componente.mean(axis=1)

# === CRIAÇÃO DO GDF ===
gdf = gpd.GeoDataFrame(
    df_asc,
    geometry=gpd.points_from_xy(df_asc.easting, df_asc.northing),
    crs="EPSG:3035"  # ou outro se o teu CRS for diferente
)
gdf_web = gdf.to_crs(epsg=3857)

# === ESCALA DE CORES ===
# Usar colormap típico de INSAR
coluna_valor = f"mean_{nome_componente}"
cmap = plt.colormaps.get_cmap('jet')  # clássico INSAR

# Escala centrada no zero
norma = colors.TwoSlopeNorm(
    vmin=gdf_web[coluna_valor].min(),
    vcenter=0,
    vmax=gdf_web[coluna_valor].max()
)

# === PLOT ===
fig, ax = plt.subplots(figsize=(8, 5))
gdf_web.plot(
    ax=ax,
    column=coluna_valor,
    cmap=cmap,
    markersize=8,
    norm=norma,
    legend=False
)

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()
plt.title(f"Deslocamento médio ({nome_componente}) - Escala INSAR", fontsize=14)

# Barra de cores
sm = ScalarMappable(cmap=cmap, norm=norma)
sm._A = []
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Deslocamento (m)", fontsize=12)

plt.show()


In [ ]:
import numpy as np
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from tslearn.clustering import TimeSeriesKMeans

# Original
serie_temporal = up_fusion.values

# Filtrar séries válidas
mask_validas = ~np.isnan(serie_temporal).all(axis=1)
serie_temporal_filtrada = serie_temporal[mask_validas]

# Normalizar
scaler = TimeSeriesScalerMeanVariance()
serie_normalizada = scaler.fit_transform(serie_temporal_filtrada)

# Clustering
n_clusters = 5
modelo = TimeSeriesKMeans(n_clusters=n_clusters, metric="dtw", random_state=0)
labels = modelo.fit_predict(serie_normalizada)

# Mapear para o GeoDataFrame completo
labels_full = -1 * np.ones(serie_temporal.shape[0], dtype=int)
labels_full[mask_validas] = labels
gdf["cluster"] = labels_full



In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx

# Primeiro, reprojetar para web mercator se ainda não fez
gdf_web = gdf.to_crs(epsg=3857)

# Fazer o plot
fig, ax = plt.subplots(figsize=(8, 6))
gdf_web[gdf_web["cluster"] != -1].plot(
    ax=ax,
    column="cluster",
    cmap="tab10",  # paleta de cores para clusters
    markersize=8,
    legend=True,
    categorical=True
)

# Adicionar o basemap
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)

# Estilizar
ax.set_axis_off()
plt.title(f"Clusters de Séries Temporais - {nome_componente}", fontsize=14)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Pegar de novo as séries
serie_temporal = up_fusion.values
tempo = up_fusion.columns  # Supondo que suas colunas são o tempo (datas ou épocas)

# Número de clusters (tirando -1)
clusters_validos = np.unique(gdf["cluster"])
clusters_validos = clusters_validos[clusters_validos != -1]

# Para cada cluster
for cluster_id in clusters_validos:
    # Seleciona as séries do cluster
    idx = gdf["cluster"] == cluster_id
    series_cluster = serie_temporal[idx.values]
    
    # Plot
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Plotar todas as séries individuais
    for serie in series_cluster:
        ax.plot(tempo, serie, color='lightgray', linewidth=0.8, alpha=0.5)

    # Plotar a média (em vermelho)
    media_cluster = np.nanmean(series_cluster, axis=0)
    ax.plot(tempo, media_cluster, color='red', linewidth=3, label='Média')

    # Estética
    ax.set_title(f"Cluster {cluster_id} - Séries Individuais + Média ({nome_componente})", fontsize=16)
    ax.set_xlabel("Tempo", fontsize=12)
    ax.set_ylabel("Deslocamento (m)", fontsize=12)
    ax.legend()
    ax.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
from shapely.geometry import MultiPoint

# Cria um multiponto a partir dos pontos
multi_point = MultiPoint(gdf_web.geometry.tolist())

# Cria o "convex hull" (fecho convexo)
hull_polygon = multi_point.convex_hull

# Cria GeoDataFrame para plotar
gdf_hull = gpd.GeoDataFrame(geometry=[hull_polygon], crs=gdf_web.crs)

# Plot
fig, ax = plt.subplots(figsize=(12, 10))
gdf_web.plot(ax=ax, color="blue", markersize=5)
gdf_hull.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=2)
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()
plt.title("Convex Hull dos Pontos")
plt.show()


In [ ]:
gdf_web['mean_up_fusion'].plot.hist(bins=50, color='skyblue')
plt.title("Distribuição dos deslocamentos médios")
plt.xlabel("Deslocamento (m)")
plt.show()


In [ ]:
from sklearn.cluster import KMeans
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

# Número de clusters (podes ajustar)
n_clusters = 5

# --- Criar GeoDataFrame ---
gdf = gpd.GeoDataFrame(
    df_asc,
    geometry=gpd.points_from_xy(df_asc.easting, df_asc.northing),
    crs="EPSG:3035"
)
gdf_web = gdf.to_crs(epsg=3857)

# --- KMeans clustering com coordenadas projetadas ---
coords = np.vstack([gdf_web.geometry.x, gdf_web.geometry.y]).T
kmeans = KMeans(n_clusters=n_clusters, random_state=0).fit(coords)
gdf_web["cluster"] = kmeans.labels_

# --- Plot ---
fig, ax = plt.subplots(figsize=(12, 10))
gdf_web.plot(ax=ax, column="cluster", categorical=True, legend=True, cmap="tab10", markersize=10)
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()
plt.title(f"Agrupamento de pontos (KMeans com {n_clusters} clusters)", fontsize=14)
plt.show()


In [ ]:
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from sklearn.cluster import KMeans
from matplotlib import cm
import matplotlib.dates as mdates

# Número de clusters
n_clusters = 5

# --- Criar GeoDataFrame ---
gdf = gpd.GeoDataFrame(
    df_asc,
    geometry=gpd.points_from_xy(df_asc.easting, df_asc.northing),
    crs="EPSG:3035"
)
gdf_web = gdf.to_crs(epsg=3857)

# --- KMeans clustering ---
coords = np.vstack([gdf_web.geometry.x, gdf_web.geometry.y]).T
kmeans = KMeans(n_clusters=n_clusters, random_state=0).fit(coords)
gdf_web["cluster"] = kmeans.labels_

# Converter os valores para datetime, assumindo o formato AAAAMMDD
tempo = pd.to_datetime(tempo.astype(str), format="%Y%m%d")

# --- Séries temporais ---
serie_temporal = up_fusion.values
tempo = up_fusion.columns

# --- Cores para médias ---
cmap = cm.get_cmap('tab10', n_clusters)
cores_clusters = [cmap(i) for i in range(n_clusters)]

# --- Layout com matplotlib gridspec ---
import matplotlib.gridspec as gridspec

# Número de linhas necessárias para os subplots
ncols = 2
nrows = int(np.ceil(n_clusters / ncols))  # arredonda para cima

fig = plt.figure(figsize=(20, 5 + 3 * nrows))
gs = gridspec.GridSpec(nrows + 1, ncols, height_ratios=[2] + [1]*nrows, hspace=0.4, wspace=0.3)

# --- Mapa na primeira linha (ocupando as 2 colunas) ---
ax_mapa = fig.add_subplot(gs[0, :])
gdf_web.plot(ax=ax_mapa, column="cluster", categorical=True, legend=True, cmap="tab10", markersize=10)
ctx.add_basemap(ax_mapa, source=ctx.providers.Esri.WorldImagery)
ax_mapa.set_axis_off()
ax_mapa.set_title(f"Clusters espaciais (KMeans, {n_clusters} clusters)", fontsize=18)

# --- Plots para cada cluster ---
clusters_validos = np.unique(gdf_web["cluster"])

for i, cluster_id in enumerate(clusters_validos):
    row = (i // ncols) + 1  # +1 por causa do mapa
    col = i % ncols
    ax = fig.add_subplot(gs[row, col])
    
    # Pegar séries do cluster
    idx = gdf_web["cluster"] == cluster_id
    series_cluster = serie_temporal[idx.values]
    
    # Plotar séries individuais (cinza)
    for serie in series_cluster:
        ax.plot(tempo, serie, color='lightgray', linewidth=0.5, alpha=0.5)
    
    # Plotar linha média
    media_cluster = np.nanmean(series_cluster, axis=0)
    ax.plot(tempo, media_cluster, color=cores_clusters[cluster_id], linewidth=3, label=f"Média Cluster {cluster_id}")
    
    ax.set_title(f"Cluster {cluster_id} ({series_cluster.shape[0]} séries)", fontsize=12)
    ax.grid(True)
    ax.legend()
    ax.set_ylabel("Deslocamento (m)")

    # Melhorar eixo de datas
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=8))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

# Eixo X apenas na última linha
for ax in fig.get_axes()[-ncols:]:
    ax.set_xlabel("Tempo", fontsize=12)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()



In [ ]:
#Sim, o código atual realiza o agrupamento (clustering) com base na localização espacial dos pontos (easting e northing).


import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from sklearn.cluster import KMeans
from matplotlib import cm
import matplotlib.dates as mdates

# Número de clusters
n_clusters = 5

# --- Criar GeoDataFrame ---
gdf = gpd.GeoDataFrame(
    df_asc,
    geometry=gpd.points_from_xy(df_asc.easting, df_asc.northing),
    crs="EPSG:3035"
)
gdf_web = gdf.to_crs(epsg=3857)

# --- KMeans clustering ---
coords = np.vstack([gdf_web.geometry.x, gdf_web.geometry.y]).T
kmeans = KMeans(n_clusters=n_clusters, random_state=0).fit(coords)
gdf_web["cluster"] = kmeans.labels_

# --- Séries temporais ---
serie_temporal = up_fusion.values
tempo = up_fusion.columns

# Converter os valores para datetime, assumindo o formato AAAAMMDD
tempo = pd.to_datetime(tempo.astype(str), format="%Y%m%d")

# --- Cores para médias ---
cmap = cm.get_cmap('tab10', n_clusters)
cores_clusters = [cmap(i) for i in range(n_clusters)]

# --- Layout com matplotlib gridspec ---
import matplotlib.gridspec as gridspec

# Definir o número de colunas e linhas para o layout
ncols = 1  # Mapa na primeira linha, gráficos nas linhas seguintes
nrows = n_clusters + 1  # Uma linha para o mapa, e n_clusters para os gráficos
fig = plt.figure(figsize=(16, 5 + 2.5 * nrows))  # Ajusta a largura para o mapa maior
gs = gridspec.GridSpec(nrows, ncols, height_ratios=[2] + [0.75]*n_clusters, wspace=0.3)

# --- Mapa na primeira linha ---
ax_mapa = fig.add_subplot(gs[0, 0])  # O mapa ocupa a primeira linha
gdf_web.plot(ax=ax_mapa, column="cluster", categorical=True, legend=True, cmap="tab10", markersize=10)
ctx.add_basemap(ax_mapa, source=ctx.providers.Esri.WorldImagery)
ax_mapa.set_axis_off()
ax_mapa.set_title(f"Clusters espaciais (KMeans, {n_clusters} clusters)", fontsize=16)

# --- Plots para cada cluster nas linhas seguintes ---
clusters_validos = np.unique(gdf_web["cluster"])

for i, cluster_id in enumerate(clusters_validos):
    ax = fig.add_subplot(gs[i + 1, 0])  # Os gráficos ocupam as linhas seguintes
    
    # Pegar séries do cluster
    idx = gdf_web["cluster"] == cluster_id
    series_cluster = serie_temporal[idx.values]
    
    # Plotar séries individuais (cinza)
    for serie in series_cluster:
        ax.plot(tempo, serie, color='lightgray', linewidth=0.5, alpha=0.5)
    
    # Plotar linha média
    media_cluster = np.nanmean(series_cluster, axis=0)
    ax.plot(tempo, media_cluster, color=cores_clusters[cluster_id], linewidth=2, label=f"Média Cluster {cluster_id}")
    
    ax.set_title(f"Cluster {cluster_id} ({series_cluster.shape[0]} séries)", fontsize=14)
    ax.grid(True)
    ax.legend()
    ax.set_ylabel("Deslocamento (m)", fontsize=12)

    # Ajustar eixo de datas
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=8))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

# Eixo X apenas na última linha
for ax in fig.get_axes()[-ncols:]:
    ax.set_xlabel("Tempo", fontsize=12)

# Ajustar rotação das datas
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
#Para realizar o agrupamento com base no comportamento das séries temporais, podemos usar algoritmos de clustering como o TimeSeriesKMeans da biblioteca tslearn, que suporta métricas como DTW (Dynamic Time Warping) para comparar séries temporais.

#Aqui está como ajustar o código para realizar o clustering com base no comportamento das séries temporais:

#Código atualizado para clustering por comportamento das séries temporais:


from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from tslearn.clustering import TimeSeriesKMeans
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import contextily as ctx
from matplotlib import cm
import matplotlib.dates as mdates
import pandas as pd

# --- Pré-processamento das séries temporais ---
# Obter as séries temporais
serie_temporal = up_fusion.values  # Supondo que up_fusion contém as séries temporais
tempo = pd.to_datetime(up_fusion.columns.astype(str), format="%Y%m%d")  # Converter colunas para datetime

# Filtrar séries válidas (sem NaN em todas as colunas)
mask_validas = ~np.isnan(serie_temporal).all(axis=1)
serie_temporal_filtrada = serie_temporal[mask_validas]

# Normalizar as séries temporais
scaler = TimeSeriesScalerMeanVariance()
serie_normalizada = scaler.fit_transform(serie_temporal_filtrada)

# --- Clustering por comportamento ---
n_clusters = 5  # Número de clusters
modelo = TimeSeriesKMeans(n_clusters=n_clusters, metric="dtw", random_state=0)
labels = modelo.fit_predict(serie_normalizada)

# Mapear os rótulos para o GeoDataFrame completo
labels_full = -1 * np.ones(serie_temporal.shape[0], dtype=int)  # Inicializar com -1
labels_full[mask_validas] = labels
gdf["cluster"] = labels_full

# --- Visualização ---
# Criar GeoDataFrame reprojetado para visualização
gdf_web = gdf.to_crs(epsg=3857)

# Cores para os clusters
cmap = cm.get_cmap('tab10', n_clusters)
cores_clusters = [cmap(i) for i in range(n_clusters)]

# Layout com matplotlib gridspec
import matplotlib.gridspec as gridspec

ncols = 1  # Mapa na primeira linha, gráficos nas linhas seguintes
nrows = n_clusters + 1  # Uma linha para o mapa, e n_clusters para os gráficos
fig = plt.figure(figsize=(16, 5 + 2.5 * nrows))
gs = gridspec.GridSpec(nrows, ncols, height_ratios=[2] + [0.75] * n_clusters, wspace=0.3)

# --- Mapa na primeira linha ---
ax_mapa = fig.add_subplot(gs[0, 0])
gdf_web.plot(ax=ax_mapa, column="cluster", categorical=True, legend=True, cmap="tab10", markersize=10)
ctx.add_basemap(ax_mapa, source=ctx.providers.Esri.WorldImagery)
ax_mapa.set_axis_off()
ax_mapa.set_title(f"Clusters por comportamento (KMeans, {n_clusters} clusters)", fontsize=16)

# --- Plots para cada cluster ---
clusters_validos = np.unique(labels)

for i, cluster_id in enumerate(clusters_validos):
    ax = fig.add_subplot(gs[i + 1, 0])
    
    # Selecionar séries do cluster
    idx = labels == cluster_id
    series_cluster = serie_temporal_filtrada[idx]
    
    # Plotar séries individuais (cinza)
    for serie in series_cluster:
        ax.plot(tempo, serie, color='lightgray', linewidth=0.5, alpha=0.5)
    
    # Plotar linha média
    media_cluster = np.nanmean(series_cluster, axis=0)
    ax.plot(tempo, media_cluster, color=cores_clusters[cluster_id], linewidth=2, label=f"Média Cluster {cluster_id}")
    
    ax.set_title(f"Cluster {cluster_id} ({series_cluster.shape[0]} séries)", fontsize=14)
    ax.grid(True)
    ax.legend()
    ax.set_ylabel("Deslocamento (m)", fontsize=12)

    # Ajustar eixo de datas
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=8))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

# Eixo X apenas na última linha
for ax in fig.get_axes()[-ncols:]:
    ax.set_xlabel("Tempo", fontsize=12)

# Ajustar rotação das datas
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


#Explicação:
#Pré-processamento das séries temporais:

#As séries temporais são normalizadas para ter média 0 e variância 1 usando TimeSeriesScalerMeanVariance.
#Apenas séries válidas (sem NaN em todas as colunas) são consideradas.
#Clustering com TimeSeriesKMeans:

#O algoritmo TimeSeriesKMeans é usado com a métrica DTW (Dynamic Time Warping), que é adequada para comparar séries temporais com diferentes padrões.
#Mapeamento dos clusters:

#Os rótulos dos clusters são atribuídos ao GeoDataFrame para visualização espacial.
#Visualização:

#O mapa mostra os clusters espaciais.
#Para cada cluster, as séries temporais individuais e a média do cluster são plotadas.
#Resultado:
#O mapa exibirá os clusters com base no comportamento das séries temporais.
#Os gráficos abaixo do mapa mostrarão as séries temporais individuais e a média de cada cluster.

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
import contextily as ctx
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np

# === CONFIGURAÇÃO ===
componente = up_fusion
nome_componente = "up_fusion"

# Adiciona média da série temporal ao DataFrame original
df_asc[f"mean_{nome_componente}"] = componente.mean(axis=1)

# === CRIAÇÃO DO GDF ===
gdf = gpd.GeoDataFrame(
    df_asc,
    geometry=gpd.points_from_xy(df_asc.easting, df_asc.northing),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# === Remover pontos com NaN na média
gdf = gdf.dropna(subset=[f"mean_{nome_componente}"])

# === CLUSTERING ===
X = np.vstack([
    gdf.geometry.x,
    gdf.geometry.y,
    gdf[f"mean_{nome_componente}"]
]).T

X_scaled = StandardScaler().fit_transform(X)

k = 5  # número de clusters
kmeans = KMeans(n_clusters=k, random_state=0, n_init="auto").fit(X_scaled)
gdf["cluster"] = kmeans.labels_

# === PLOT ===
fig, ax = plt.subplots(figsize=(12, 10))
gdf.plot(
    ax=ax,
    column="cluster",
    categorical=True,
    cmap="tab10",
    markersize=10,
    legend=True
)
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()
plt.title(f"Clusters baseados em localização + deslocamento médio ({nome_componente})", fontsize=14)
plt.show()



In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
from sklearn.cluster import KMeans
import contextily as ctx
import numpy as np

# === CONFIGURAÇÃO ===
componente = up_fusion
nome_componente = "up_fusion"

# === MÉDIA DO DESLOCAMENTO ===
df_asc[f"mean_{nome_componente}"] = componente.mean(axis=1)

# === CRIAÇÃO DO GeoDataFrame ===
gdf = gpd.GeoDataFrame(
    df_asc,
    geometry=gpd.points_from_xy(df_asc.easting, df_asc.northing),
    crs="EPSG:3035"
)
gdf_web = gdf.to_crs(epsg=3857)

# === PREPARAR DADOS PARA KMEANS ===
coluna_valor = f"mean_{nome_componente}"
gdf_kmeans = gdf_web.dropna(subset=[coluna_valor]).copy()

X = gdf_kmeans[[coluna_valor]].values  # Apenas deslocamento

# === APLICAR KMEANS ===
n_clusters = 4  # podes ajustar conforme quiseres
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
gdf_kmeans["cluster"] = kmeans.fit_predict(X)

# === VISUALIZAÇÃO ===
fig, ax = plt.subplots(figsize=(12, 10))

gdf_kmeans.plot(
    ax=ax,
    column="cluster",
    categorical=True,
    cmap="Set1",
    markersize=10,
    legend=True
)

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()
ax.set_title(f"KMeans baseado apenas no deslocamento médio ({coluna_valor})", fontsize=14)

plt.show()




In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
from sklearn.cluster import KMeans
import contextily as ctx
import numpy as np

# === CONFIGURAÇÃO ===
componente = up_fusion
nome_componente = "up_fusion"

# === MÉDIA DO DESLOCAMENTO ===
df_asc[f"mean_{nome_componente}"] = componente.mean(axis=1)

# === CRIAÇÃO DO GeoDataFrame ===
gdf = gpd.GeoDataFrame(
    df_asc,
    geometry=gpd.points_from_xy(df_asc.easting, df_asc.northing),
    crs="EPSG:3035"
)
gdf_web = gdf.to_crs(epsg=3857)

# === PREPARAR DADOS PARA KMEANS ===
coluna_valor = f"mean_{nome_componente}"
gdf_kmeans = gdf_web.dropna(subset=[coluna_valor]).copy()

X = gdf_kmeans[[coluna_valor]].values  # Apenas deslocamento

# === APLICAR KMEANS ===
n_clusters = 4  # Ajuste conforme necessário
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
gdf_kmeans["cluster"] = kmeans.fit_predict(X)

# === VISUALIZAÇÃO ===
fig, ax = plt.subplots(figsize=(12, 10))

# Usar colormap típico do InSAR (jet)
cmap = plt.cm.jet
norm = colors.Normalize(vmin=gdf_kmeans[coluna_valor].min(), vmax=gdf_kmeans[coluna_valor].max())

# Plotar os clusters com o colormap do InSAR
gdf_kmeans.plot(
    ax=ax,
    column="cluster",
    cmap=cmap,
    markersize=10,
    legend=False  # Desativar a legenda automática
)

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()
ax.set_title(f"KMeans baseado apenas no deslocamento médio ({coluna_valor})", fontsize=14)

# === Criar legenda personalizada para cada cluster ===
handles = []
for cluster_id in range(n_clusters):
    cluster_data = gdf_kmeans[gdf_kmeans["cluster"] == cluster_id]
    cluster_min = cluster_data[coluna_valor].min()
    cluster_max = cluster_data[coluna_valor].max()
    handles.append(plt.Line2D(
        [0], [0], marker='o', color='w',
        label=f"Cluster {cluster_id}: {cluster_min:.2f} a {cluster_max:.2f} m",
        markerfacecolor=cmap(cluster_id / n_clusters), markersize=10
    ))

# Adicionar a legenda ao gráfico
ax.legend(handles=handles, loc='upper left', title="Clusters e Limites", fontsize=10)

# Adicionar barra de cores para o deslocamento
sm = ScalarMappable(cmap=cmap, norm=norm)
sm._A = []  # Necessário para ScalarMappable funcionar
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label(f"{coluna_valor} (m)", fontsize=12)

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

# Exemplo de dados simulados
# Supondo que 'up_fusion' tenha colunas como datas e valores de deslocamento
dates = pd.date_range(start="2020-01-01", periods=36, freq="M")
data = {
    "point_1": np.sin(np.linspace(0, 3 * np.pi, 36)) + np.random.normal(0, 0.1, 36),
    "point_2": np.cos(np.linspace(0, 3 * np.pi, 36)) + np.random.normal(0, 0.1, 36),
}
up_fusion = pd.DataFrame(data, index=dates)

# Escolher um ponto para decomposição (exemplo: 'point_1')
serie = up_fusion["point_1"]

# Preencher valores ausentes (necessário para o seasonal_decompose)
serie = serie.interpolate(method="linear")

# Decomposição usando seasonal_decompose
result = seasonal_decompose(serie, model="additive", period=12)

# Plotar os componentes
fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)

axes[0].plot(serie, label="Original")
axes[0].set_title("Série Original")
axes[0].legend()

axes[1].plot(result.trend, label="Tendência", color="red")
axes[1].set_title("Tendência")
axes[1].legend()

axes[2].plot(result.seasonal, label="Sazonalidade", color="blue")
axes[2].set_title("Sazonalidade")
axes[2].legend()

axes[3].plot(result.resid, label="Resíduos", color="green")
axes[3].set_title("Resíduos")
axes[3].legend()

plt.tight_layout()
plt.show()

In [ ]:
df_nivel = pd.read_excel("data/nivel.xlsx")
df_nivel

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
from mpl_toolkits.axes_grid1 import make_axes_locatable
import seaborn as sns
import numpy as np

# ---------- CÁLCULO DA MÉDIA VERTICAL ----------
df_asc["mean_up_fusion"] = up_fusion.mean(axis=1)

# Colunas temporais e datas
colunas_temporais = [col for col in up_fusion.columns if col.startswith("20")]
datas = pd.to_datetime(colunas_temporais)

# ---------- PREPARAR GEO ----------
gdf_all = gpd.GeoDataFrame(
    df_asc,
    geometry=gpd.points_from_xy(df_asc.easting, df_asc.northing),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# Escala de cores
norm = colors.TwoSlopeNorm(
    vmin=df_asc["mean_up_fusion"].min(),
    vcenter=0,
    vmax=df_asc["mean_up_fusion"].max()
)

# Limites do mapa
xmin, ymin, xmax, ymax = gdf_all.total_bounds

# ---------- CALCULAR A CORRELAÇÃO PARA TODOS OS PONTOS ----------
correlations = {}

# Prepara o df_nivel
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel.set_index('data', inplace=True)

for idx in up_fusion.index:
    serie_up = up_fusion.loc[idx, colunas_temporais].astype(float)
    nivel_data = df_nivel.reindex(datas, method='nearest')
    nivel_data_values = nivel_data['nivel'].values

    # Remover NaN
    valid_data = ~np.isnan(serie_up) & ~np.isnan(nivel_data_values)
    if valid_data.sum() > 1:
        corr = np.corrcoef(serie_up[valid_data], nivel_data_values[valid_data])[0,1]
        correlations[idx] = corr

# Converter dicionário em Series
correlations = pd.Series(correlations)

# Pegar o ponto com maior correlação (em módulo)
idx_max_corr = correlations.abs().idxmax()
corr_value = correlations[idx_max_corr]

print(f"Ponto com maior correlação: {idx_max_corr}, Correlação = {corr_value:.2f}")

# ---------- PREPARAR OS DADOS DESSE PONTO ----------
row = df_asc.loc[idx_max_corr]
gdf_row = gpd.GeoDataFrame(
    pd.DataFrame([row]),
    geometry=gpd.points_from_xy([row.easting], [row.northing]),
    crs="EPSG:3035"
).to_crs(epsg=3857)

serie_up_fusion = up_fusion.loc[idx_max_corr, colunas_temporais].astype(float)
nivel_data = df_nivel.reindex(datas, method='nearest')
nivel_data_values = nivel_data['nivel'].values

# Limpar NaNs
valid_data = ~np.isnan(serie_up_fusion) & ~np.isnan(nivel_data_values)
serie_up_fusion_clean = serie_up_fusion[valid_data]
nivel_data_values_clean = nivel_data_values[valid_data]


# ---------- PLOTAR MAPA + SÉRIE TEMPORAL ----------
fig, axs = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={'width_ratios': [1.2, 1]})
fig.subplots_adjust(hspace=0.5, wspace=0.3)

# Mapa
ax_map = axs[0]
divider = make_axes_locatable(ax_map)
cax = divider.append_axes("right", size="5%", pad=0.05)

gdf_all.plot(
    ax=ax_map,
    column="mean_up_fusion",
    cmap="jet",
    markersize=5,
    norm=norm,
    legend=True,
    cax=cax
)
gdf_row.plot(
    ax=ax_map,
    color="red",
    edgecolor="black",
    markersize=100
)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

ax_map.set_xlim(xmin, xmax)
ax_map.set_ylim(ymin, ymax)
ax_map.set_title(f"PID {row['pid']} | Média: {row['mean_up_fusion']:.2f} m")
ax_map.set_axis_off()

# Séries temporais
ax_series = axs[1]
ax_series.plot(datas, serie_up_fusion, color="blue", label="up_fusion")
ax_series.axhline(0, color='gray', linestyle='--', linewidth=0.7)
ax_series.set_title("Série Temporal: up_fusion vs Nível")
ax_series.set_ylabel("Deslocamento (m)")
ax_series.set_xlabel("Data")
ax_series.grid(True)

if len(nivel_data) > 0:
    ax2 = ax_series.twinx()
    ax2.plot(datas, nivel_data['nivel'], color="green", label="Nível")
    ax2.set_ylabel("Nível (m)", color="green")
    ax2.tick_params(axis='y', labelcolor="green")
    ax_series.legend(loc='upper left')
    ax2.legend(loc='upper right')
else:
    print("Nenhum dado de nível disponível.")

plt.tight_layout()
plt.show()

# ---------- PLOTAR DISPERSÃO ----------
plt.figure(figsize=(8, 6))
sns.scatterplot(x=serie_up_fusion_clean, y=nivel_data_values_clean, color="blue", label="Pontos")

# Linha de regressão
sns.regplot(x=serie_up_fusion_clean, y=nivel_data_values_clean, scatter=False, color="red", line_kws={"lw": 2, "ls": "--"})

plt.title(f'Correlação entre up_fusion e Nível (Correlação = {corr_value:.2f})')
plt.xlabel('Deslocamento (m) - up_fusion')
plt.ylabel('Nível (m)')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.gridspec as gridspec

# ------- Layout melhorado para mapa + série temporal + dispersão -------
fig = plt.figure(figsize=(18, 10))  # Tamanho maior
gs = gridspec.GridSpec(2, 2, height_ratios=[2, 1], width_ratios=[1.3, 1])

# Mapa
ax_map = fig.add_subplot(gs[0, 0])
divider = make_axes_locatable(ax_map)
cax = divider.append_axes("right", size="5%", pad=0.05)

gdf_all.plot(
    ax=ax_map,
    column="mean_up_fusion",
    cmap="jet",
    markersize=5,
    norm=norm,
    legend=True,
    cax=cax
)
gdf_row.plot(
    ax=ax_map,
    color="red",
    edgecolor="black",
    markersize=100
)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

ax_map.set_xlim(xmin, xmax)
ax_map.set_ylim(ymin, ymax)
ax_map.set_title(f"PID {row['pid']} | Média: {row['mean_up_fusion']:.2f} m")
ax_map.set_axis_off()

# Série Temporal
ax_series = fig.add_subplot(gs[0, 1])
ax_series.plot(datas, serie_up_fusion, color="blue", label="up_fusion")
ax_series.axhline(0, color='gray', linestyle='--', linewidth=0.7)
ax_series.set_title("Série Temporal: up_fusion vs Nível")
ax_series.set_ylabel("Deslocamento (m)")
ax_series.set_xlabel("Data")
ax_series.grid(True)

if len(nivel_data) > 0:
    ax2 = ax_series.twinx()
    ax2.plot(datas, nivel_data['nivel'], color="green", label="Nível")
    ax2.set_ylabel("Nível (m)", color="green")
    ax2.tick_params(axis='y', labelcolor="green")
    ax_series.legend(loc='upper left')
    ax2.legend(loc='upper right')
else:
    print("Nenhum dado de nível disponível.")

# Dispersão Correlação
ax_corr = fig.add_subplot(gs[1, :])
sns.scatterplot(x=serie_up_fusion_clean, y=nivel_data_values_clean, color="blue", label="Pontos", ax=ax_corr)
sns.regplot(x=serie_up_fusion_clean, y=nivel_data_values_clean, scatter=False, color="red", line_kws={"lw": 2, "ls": "--"}, ax=ax_corr)

ax_corr.set_title(f'Correlação entre up_fusion e Nível (Correlação = {corr_value:.2f})')
ax_corr.set_xlabel('Deslocamento (m) - up_fusion')
ax_corr.set_ylabel('Nível (m)')
ax_corr.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import statsmodels.api as sm  # Importar para a técnica de LOWESS

# ------- Layout melhorado para mapa + série temporal + dispersão -------
fig = plt.figure(figsize=(18, 10))  # Tamanho maior
gs = gridspec.GridSpec(2, 2, height_ratios=[2, 1], width_ratios=[1.3, 1])

# Mapa
ax_map = fig.add_subplot(gs[0, 0])
divider = make_axes_locatable(ax_map)
cax = divider.append_axes("right", size="5%", pad=0.05)

gdf_all.plot(
    ax=ax_map,
    column="mean_up_fusion",
    cmap="jet",
    markersize=5,
    norm=norm,
    legend=True,
    cax=cax
)
gdf_row.plot(
    ax=ax_map,
    color="red",
    edgecolor="black",
    markersize=100
)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

ax_map.set_xlim(xmin, xmax)
ax_map.set_ylim(ymin, ymax)
ax_map.set_title(f"PID {row['pid']} | Média: {row['mean_up_fusion']:.2f} m")
ax_map.set_axis_off()

# Série Temporal
ax_series = fig.add_subplot(gs[0, 1])
ax_series.plot(datas, serie_up_fusion, color="blue", label="up_fusion")
ax_series.axhline(0, color='gray', linestyle='--', linewidth=0.7)

# Aplicar LOWESS para suavizar a série temporal
lowess = sm.nonparametric.lowess
smoothed = lowess(serie_up_fusion.values, datas.values.astype(float), frac=0.2)  # frac controla o grau de suavização
ax_series.plot(pd.to_datetime(smoothed[:, 0]), smoothed[:, 1], color="red", linestyle="--", linewidth=2, label="Tendência (LOWESS)")

ax_series.set_title("Série Temporal: up_fusion vs Nível")
ax_series.set_ylabel("Deslocamento (m)")
ax_series.set_xlabel("Data")
ax_series.grid(True)

if len(nivel_data) > 0:
    ax2 = ax_series.twinx()
    ax2.plot(datas, nivel_data['nivel'], color="green", label="Nível")
    ax2.set_ylabel("Nível (m)", color="green")
    ax2.tick_params(axis='y', labelcolor="green")
    ax_series.legend(loc='upper left')
    ax2.legend(loc='upper right')
else:
    print("Nenhum dado de nível disponível.")

# Dispersão Correlação
ax_corr = fig.add_subplot(gs[1, :])
sns.scatterplot(x=serie_up_fusion_clean, y=nivel_data_values_clean, color="blue", label="Pontos", ax=ax_corr)
sns.regplot(x=serie_up_fusion_clean, y=nivel_data_values_clean, scatter=False, color="red", line_kws={"lw": 2, "ls": "--"}, ax=ax_corr)

ax_corr.set_title(f'Correlação entre up_fusion e Nível (Correlação = {corr_value:.2f})')
ax_corr.set_xlabel('Deslocamento (m) - up_fusion')
ax_corr.set_ylabel('Nível (m)')
ax_corr.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.gridspec as gridspec
import statsmodels.api as sm  # <-- importar para a tendência

# ------- Layout melhorado para mapa + série temporal + dispersão -------
fig = plt.figure(figsize=(18, 10))  # Tamanho maior
gs = gridspec.GridSpec(2, 2, height_ratios=[2, 1], width_ratios=[1.3, 1])

# Mapa
ax_map = fig.add_subplot(gs[0, 0])
divider = make_axes_locatable(ax_map)
cax = divider.append_axes("right", size="5%", pad=0.05)

gdf_all.plot(
    ax=ax_map,
    column="mean_up_fusion",
    cmap="jet",
    markersize=5,
    norm=norm,
    legend=True,
    cax=cax
)
gdf_row.plot(
    ax=ax_map,
    color="red",
    edgecolor="black",
    markersize=100
)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

ax_map.set_xlim(xmin, xmax)
ax_map.set_ylim(ymin, ymax)
ax_map.set_title(f"PID {row['pid']} | Média: {row['mean_up_fusion']:.2f} m")
ax_map.set_axis_off()

# Série Temporal
ax_series = fig.add_subplot(gs[0, 1])
ax_series.plot(datas, serie_up_fusion, color="blue", label="up_fusion")
ax_series.axhline(0, color='gray', linestyle='--', linewidth=0.7)

# --- Adicionar curva de tendência (LOWESS) ---
lowess = sm.nonparametric.lowess
smoothed = lowess(serie_up_fusion.values, datas.values.astype(float), frac=0.2)  # datas precisam estar em float para LOWESS
ax_series.plot(pd.to_datetime(smoothed[:, 0]), smoothed[:, 1], color="red", linestyle="--", linewidth=2, label="Tendência")
# ------------------------------------------------

ax_series.set_title("Série Temporal: up_fusion vs Nível")
ax_series.set_ylabel("Deslocamento (m)")
ax_series.set_xlabel("Data")
ax_series.grid(True)

if len(nivel_data) > 0:
    ax2 = ax_series.twinx()
    ax2.plot(datas, nivel_data['nivel'], color="green", label="Nível")
    ax2.set_ylabel("Nível (m)", color="green")
    ax2.tick_params(axis='y', labelcolor="green")
    ax_series.legend(loc='upper left')
    ax2.legend(loc='upper right')
else:
    print("Nenhum dado de nível disponível.")

# Dispersão Correlação
ax_corr = fig.add_subplot(gs[1, :])
sns.scatterplot(x=serie_up_fusion_clean, y=nivel_data_values_clean, color="blue", label="Pontos", ax=ax_corr)
sns.regplot(x=serie_up_fusion_clean, y=nivel_data_values_clean, scatter=False, color="red", line_kws={"lw": 2, "ls": "--"}, ax=ax_corr)

ax_corr.set_title(f'Correlação entre up_fusion e Nível (Correlação = {corr_value:.2f})')
ax_corr.set_xlabel('Deslocamento (m) - up_fusion')
ax_corr.set_ylabel('Nível (m)')
ax_corr.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.gridspec as gridspec

# ------- Layout melhorado para mapa + série temporal + dispersão -------
fig = plt.figure(figsize=(18, 10))  # Tamanho maior
gs = gridspec.GridSpec(2, 2, height_ratios=[2, 1], width_ratios=[1.3, 1])

# Mapa
ax_map = fig.add_subplot(gs[0, 0])
divider = make_axes_locatable(ax_map)
cax = divider.append_axes("right", size="5%", pad=0.05)

gdf_all.plot(
    ax=ax_map,
    column="mean_up_fusion",
    cmap="jet",
    markersize=5,
    norm=norm,
    legend=True,
    cax=cax
)
gdf_row.plot(
    ax=ax_map,
    color="red",
    edgecolor="black",
    markersize=100
)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

ax_map.set_xlim(xmin, xmax)
ax_map.set_ylim(ymin, ymax)
ax_map.set_title(f"PID {row['pid']} | Média: {row['mean_up_fusion']:.2f} m")
ax_map.set_axis_off()

import statsmodels.api as sm

# [...] (resto igual até ao ax_series)

# Série Temporal
ax_series = fig.add_subplot(gs[0, 1])
ax_series.plot(datas, serie_up_fusion, color="blue", label="up_fusion")
ax_series.axhline(0, color='gray', linestyle='--', linewidth=0.7)

# --- Adicionar curva de tendência (LOWESS) ---
lowess = sm.nonparametric.lowess
smoothed = lowess(serie_up_fusion.values, datas.values, frac=0.2)  # frac controla o quanto suaviza (0.2 = suave)

ax_series.plot(pd.to_datetime(smoothed[:, 0]), smoothed[:, 1], color="red", linestyle="--", linewidth=2, label="Tendência")

# --- Fim da tendência ---

ax_series.set_title("Série Temporal: up_fusion vs Nível")
ax_series.set_ylabel("Deslocamento (m)")
ax_series.set_xlabel("Data")
ax_series.grid(True)

if len(nivel_data) > 0:
    ax2 = ax_series.twinx()
    ax2.plot(datas, nivel_data['nivel'], color="green", label="Nível")
    ax2.set_ylabel("Nível (m)", color="green")
    ax2.tick_params(axis='y', labelcolor="green")
    ax_series.legend(loc='upper left')
    ax2.legend(loc='upper right')
else:
    print("Nenhum dado de nível disponível.")


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ---------- CÁLCULO DA MÉDIA VERTICAL ----------
df_asc["mean_up_fusion"] = up_fusion.mean(axis=1)

# Seleciona o ponto com maior deslocamento vertical médio (exemplo: o primeiro ponto da lista top10)
top1 = df_asc.nlargest(1, "mean_up_fusion").copy()
top1_gdf = gpd.GeoDataFrame(
    top1,
    geometry=gpd.points_from_xy(top1.easting, top1.northing),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# Colunas temporais e datas
colunas_temporais = [col for col in up_fusion.columns if col.startswith("20")]
datas = pd.to_datetime(colunas_temporais)

# ---------- ESCALA DE CORES CONSISTENTE ----------
norm = colors.TwoSlopeNorm(
    vmin=df_asc["mean_up_fusion"].min(),
    vcenter=0,
    vmax=df_asc["mean_up_fusion"].max()
)

# ---------- Criar um GeoDataFrame para todos os pontos ----------
gdf_all = gpd.GeoDataFrame(
    df_asc,
    geometry=gpd.points_from_xy(df_asc.easting, df_asc.northing),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# Encontrar os limites máximos para o mapa
xmin, ymin, xmax, ymax = gdf_all.total_bounds

# ---------- FIGURA COM SUBPLOTS ----------

# Ajuste o tamanho da figura
fig, axs = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={'width_ratios': [1.2, 1]})

# Ajuste o espaçamento entre os subgráficos
fig.subplots_adjust(hspace=0.5, wspace=0.3)

# Pegando o primeiro ponto de top1 para visualização
idx, row = top1.iloc[0].name, top1.iloc[0]

gdf_row = top1_gdf.loc[[idx]]  # GeoDataFrame de um ponto
ax_map = axs[0]
ax_series = axs[1]

# --- Mapa ---
divider = make_axes_locatable(ax_map)
cax = divider.append_axes("right", size="5%", pad=0.05)

gdf_all.plot(
    ax=ax_map,
    column="mean_up_fusion",
    cmap="jet",
    markersize=5,
    norm=norm,
    legend=True,
    cax=cax
)
gdf_row.plot(
    ax=ax_map,
    color="red",
    edgecolor="black",
    markersize=100
)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# Definindo o limite do mapa para mostrar todos os pontos
ax_map.set_xlim(xmin, xmax)
ax_map.set_ylim(ymin, ymax)

ax_map.set_title(f"PID {row['pid']} | Média: {row['mean_up_fusion']:.2f} m")
ax_map.set_axis_off()

# --- Série temporal ---
serie = up_fusion.loc[idx, colunas_temporais].astype(float)
ax_series.plot(datas, serie, color="blue", label=f"PID {row['pid']}")
ax_series.axhline(0, color='gray', linestyle='--', linewidth=0.7)
ax_series.set_title("Série temporal de up_fusion")
ax_series.set_ylabel("Deslocamento (m)")
ax_series.set_xlabel("Data")
ax_series.grid(True)
ax_series.legend()

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ---------- CÁLCULO DA MÉDIA VERTICAL ----------
df_asc["mean_up_fusion"] = up_fusion.mean(axis=1)

# Seleciona o ponto com maior deslocamento vertical médio (exemplo: o primeiro ponto da lista top10)
top1 = df_asc.nlargest(1, "mean_up_fusion").copy()
top1_gdf = gpd.GeoDataFrame(
    top1,
    geometry=gpd.points_from_xy(top1.easting, top1.northing),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# Colunas temporais e datas
colunas_temporais = [col for col in up_fusion.columns if col.startswith("20")]
datas = pd.to_datetime(colunas_temporais)

# ---------- ESCALA DE CORES CONSISTENTE ----------
norm = colors.TwoSlopeNorm(
    vmin=df_asc["mean_up_fusion"].min(),
    vcenter=0,
    vmax=df_asc["mean_up_fusion"].max()
)

# ---------- Criar um GeoDataFrame para todos os pontos ----------
gdf_all = gpd.GeoDataFrame(
    df_asc,
    geometry=gpd.points_from_xy(df_asc.easting, df_asc.northing),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# Encontrar os limites máximos para o mapa
xmin, ymin, xmax, ymax = gdf_all.total_bounds

# ---------- FIGURA COM SUBPLOTS ----------

# Ajuste o tamanho da figura
fig, axs = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={'width_ratios': [1.2, 1]})

# Ajuste o espaçamento entre os subgráficos
fig.subplots_adjust(hspace=0.5, wspace=0.3)

# Pegando o primeiro ponto de top1 para visualização
idx, row = top1.iloc[0].name, top1.iloc[0]

gdf_row = top1_gdf.loc[[idx]]  # GeoDataFrame de um ponto
ax_map = axs[0]
ax_series = axs[1]

# --- Mapa ---
divider = make_axes_locatable(ax_map)
cax = divider.append_axes("right", size="5%", pad=0.05)

gdf_all.plot(
    ax=ax_map,
    column="mean_up_fusion",
    cmap="jet",
    markersize=5,
    norm=norm,
    legend=True,
    cax=cax
)
gdf_row.plot(
    ax=ax_map,
    color="red",
    edgecolor="black",
    markersize=100
)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# Definindo o limite do mapa para mostrar todos os pontos
ax_map.set_xlim(xmin, xmax)
ax_map.set_ylim(ymin, ymax)

ax_map.set_title(f"PID {row['pid']} | Média: {row['mean_up_fusion']:.2f} m")
ax_map.set_axis_off()

# --- Série temporal de up_fusion ---
serie_up_fusion = up_fusion.loc[idx, colunas_temporais].astype(float)
ax_series.plot(datas, serie_up_fusion, color="blue", label=f"PID {row['pid']} (up_fusion)")
ax_series.axhline(0, color='gray', linestyle='--', linewidth=0.7)
ax_series.set_title("Série temporal de up_fusion")
ax_series.set_ylabel("Deslocamento (m)")
ax_series.set_xlabel("Data")
ax_series.grid(True)

# --- Série temporal de nível (df_nivel) ---
# Certifique-se de que a coluna 'data' de df_nivel está no mesmo formato de datetime
df_nivel['data'] = pd.to_datetime(df_nivel['data'])

# Interpola os valores de nível para as datas de up_fusion (usando 'nearest')
df_nivel.set_index('data', inplace=True)
nivel_data = df_nivel.reindex(datas, method='nearest')  # Reindexa para as datas de up_fusion

if len(nivel_data) > 0:
    ax2 = ax_series.twinx()  # Cria um segundo eixo y
    ax2.plot(nivel_data.index, nivel_data['nivel'], color="green", label="Nível (df_nivel)")
    ax2.set_ylabel("Nível (m)", color="green")
    ax2.tick_params(axis='y', labelcolor="green")

    # Ajusta a visualização e as legendas
    ax_series.legend(loc='upper left')
    ax2.legend(loc='upper right')
else:
    print("Nenhuma data de nível coincide com as datas de up_fusion.")

plt.tight_layout()
plt.show()



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Certifique-se de que as séries temporais de up_fusion e nivel tenham a mesma estrutura de datas
# Já temos 'nivel_data' que foi interpolado para as mesmas datas de 'up_fusion'
serie_up_fusion = up_fusion.loc[idx, colunas_temporais].astype(float)
nivel_data_values = nivel_data['nivel'].values

# Verifique se há valores NaN nas séries
print("Valores NaN em up_fusion:", serie_up_fusion.isna().sum())
print("Valores NaN em nivel_data:", pd.Series(nivel_data_values).isna().sum())

# Remover valores NaN (caso existam)
valid_data = ~np.isnan(serie_up_fusion) & ~np.isnan(nivel_data_values)
serie_up_fusion_clean = serie_up_fusion[valid_data]
nivel_data_values_clean = nivel_data_values[valid_data]

# Verifique novamente se há valores NaN
print("Valores NaN após limpeza:")
print("Valores NaN em up_fusion:", serie_up_fusion_clean.isna().sum())
print("Valores NaN em nivel_data:", pd.Series(nivel_data_values_clean).isna().sum())

# Calcular a correlação, se as séries não tiverem sido completamente limpas
if len(serie_up_fusion_clean) > 1:
    correlation = np.corrcoef(serie_up_fusion_clean, nivel_data_values_clean)[0, 1]
else:
    correlation = np.nan  # Se não houver dados suficientes para calcular a correlação

# Exibe a correlação
print(f"Coeficiente de correlação: {correlation}")

# Cria um gráfico de dispersão para visualizar a correlação
plt.figure(figsize=(8, 6))
sns.scatterplot(x=serie_up_fusion_clean, y=nivel_data_values_clean, color="blue", label="Pontos")

# Adiciona uma linha de tendência (regressão linear)
sns.regplot(x=serie_up_fusion_clean, y=nivel_data_values_clean, scatter=False, color="red", line_kws={"lw": 2, "ls": "--"})

# Título e rótulos do gráfico
plt.title(f'Correlação entre up_fusion e Nível (Coeficiente de correlação = {correlation:.2f})')
plt.xlabel('Deslocamento (m) - up_fusion')
plt.ylabel('Nível (m)')

# Exibe a grade e o gráfico
plt.grid(True)
plt.tight_layout()
plt.show()



In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
from mpl_toolkits.axes_grid1 import make_axes_locatable
import seaborn as sns
import numpy as np

# ---------- CÁLCULO DA MÉDIA VERTICAL ----------
df_asc["mean_up_fusion"] = up_fusion.mean(axis=1)

# Seleciona os 10 pontos com maior deslocamento vertical médio
top10 = df_asc.nlargest(10, "mean_up_fusion").copy()
top10_gdf = gpd.GeoDataFrame(
    top10,
    geometry=gpd.points_from_xy(top10.easting, top10.northing),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# Colunas temporais e datas
colunas_temporais = [col for col in up_fusion.columns if col.startswith("20")]
datas = pd.to_datetime(colunas_temporais)

# ---------- ESCALA DE CORES CONSISTENTE ----------
norm = colors.TwoSlopeNorm(
    vmin=df_asc["mean_up_fusion"].min(),
    vcenter=0,
    vmax=df_asc["mean_up_fusion"].max()
)

# ---------- Criar um GeoDataFrame para todos os pontos ----------
gdf_all = gpd.GeoDataFrame(
    df_asc,
    geometry=gpd.points_from_xy(df_asc.easting, df_asc.northing),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# Encontrar os limites máximos para o mapa
xmin, ymin, xmax, ymax = gdf_all.total_bounds

# ---------- FIGURA COM SUBPLOTS ----------
# Ajuste o tamanho da figura
for i, (idx, row) in enumerate(top10.iterrows()):
    fig, axs = plt.subplots(3, 1, figsize=(15, 15), gridspec_kw={'height_ratios': [1.5, 1, 1]})
    
    # Mapa (primeiro subgráfico)
    ax_map = axs[0]
    divider = make_axes_locatable(ax_map)
    cax = divider.append_axes("right", size="5%", pad=0.05)

    gdf_all.plot(
        ax=ax_map,
        column="mean_up_fusion",
        cmap="jet",
        markersize=5,
        norm=norm,
        legend=True,
        cax=cax
    )
    gdf_row = top10_gdf.loc[[idx]]
    gdf_row.plot(
        ax=ax_map,
        color="red",
        edgecolor="black",
        markersize=100
    )
    ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

    ax_map.set_xlim(xmin, xmax)
    ax_map.set_ylim(ymin, ymax)
    ax_map.set_title(f"PID {row['pid']} | Média: {row['mean_up_fusion']:.2f} m")
    ax_map.set_axis_off()

    # Série Temporal (segundo subgráfico)
    ax_series = axs[1]
    serie_up_fusion = up_fusion.loc[idx, colunas_temporais].astype(float)
    ax_series.plot(datas, serie_up_fusion, color="blue", label=f"PID {row['pid']} (up_fusion)")
    ax_series.axhline(0, color='gray', linestyle='--', linewidth=0.7)
    ax_series.set_title("Série Temporal de up_fusion")
    ax_series.set_ylabel("Deslocamento (m)")
    ax_series.set_xlabel("Data")
    ax_series.grid(True)

    # Série Temporal de Nível (usando eixo y secundário)
    df_nivel['data'] = pd.to_datetime(df_nivel['data'])
    df_nivel.set_index('data', inplace=True)
    nivel_data = df_nivel.reindex(datas, method='nearest')
    nivel_data_values = nivel_data.loc[datas, 'nivel'].values

    if len(nivel_data_values) > 0:
        ax2 = ax_series.twinx()  # Cria um segundo eixo y
        ax2.plot(datas, nivel_data_values, color="green", label="Nível (df_nivel)")
        ax2.set_ylabel("Nível (m)", color="green")
        ax2.tick_params(axis='y', labelcolor="green")
        ax_series.legend(loc='upper left')
        ax2.legend(loc='upper right')

    # Gráfico de Dispersão e Correlação (terceiro subgráfico)
    ax_corr = axs[2]
    valid_data = ~np.isnan(serie_up_fusion) & ~np.isnan(nivel_data_values)
    serie_up_fusion_clean = serie_up_fusion[valid_data]
    nivel_data_values_clean = nivel_data_values[valid_data]

    if len(serie_up_fusion_clean) > 1:
        correlation = np.corrcoef(serie_up_fusion_clean, nivel_data_values_clean)[0, 1]
    else:
        correlation = np.nan

    # Gráfico de dispersão
    sns.scatterplot(x=serie_up_fusion_clean, y=nivel_data_values_clean, color="blue", ax=ax_corr, label="Pontos")
    sns.regplot(x=serie_up_fusion_clean, y=nivel_data_values_clean, scatter=False, color="red", ax=ax_corr, line_kws={"lw": 2, "ls": "--"})
    
    ax_corr.set_title(f"Correlação entre up_fusion e Nível (PID {row['pid']}) | Correlação: {correlation:.2f}")
    ax_corr.set_xlabel('Deslocamento (m) - up_fusion')
    ax_corr.set_ylabel('Nível (m)')
    ax_corr.grid(True)

    # Ajustes gerais
    plt.tight_layout()
    plt.show()



In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
from mpl_toolkits.axes_grid1 import make_axes_locatable
from statsmodels.tsa.seasonal import seasonal_decompose

# ---------- CÁLCULO DA MÉDIA VERTICAL ----------
df_asc["mean_up_fusion"] = up_fusion.mean(axis=1)

# Seleciona os 10 pontos com maior deslocamento vertical médio
top10 = df_asc.nlargest(10, "mean_up_fusion").copy()
top10_gdf = gpd.GeoDataFrame(
    top10,
    geometry=gpd.points_from_xy(top10.easting, top10.northing),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# Colunas temporais e datas
colunas_temporais = [col for col in up_fusion.columns if col.startswith("20")]
datas = pd.to_datetime(colunas_temporais)

# ---------- ESCALA DE CORES CONSISTENTE ----------
norm = colors.TwoSlopeNorm(
    vmin=df_asc["mean_up_fusion"].min(),
    vcenter=0,
    vmax=df_asc["mean_up_fusion"].max()
)

# ---------- GeoDataFrame com todos os pontos ----------
gdf_all = gpd.GeoDataFrame(
    df_asc,
    geometry=gpd.points_from_xy(df_asc.easting, df_asc.northing),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# Limites do mapa
xmin, ymin, xmax, ymax = gdf_all.total_bounds

# ---------- FIGURA COM SUBPLOTS ----------
fig, axs = plt.subplots(10, 3, figsize=(20, 50), gridspec_kw={'width_ratios': [1.2, 1, 1]})
fig.subplots_adjust(hspace=0.5, wspace=0.3)

for i, (idx, row) in enumerate(top10.iterrows()):
    gdf_row = top10_gdf.loc[[idx]]
    ax_map = axs[i, 0]
    ax_series = axs[i, 1]
    ax_trend = axs[i, 2]

    # --- Mapa ---
    divider = make_axes_locatable(ax_map)
    cax = divider.append_axes("right", size="5%", pad=0.05)

    gdf_all.plot(
        ax=ax_map,
        column="mean_up_fusion",
        cmap="jet",
        markersize=5,
        norm=norm,
        legend=True,
        cax=cax
    )
    gdf_row.plot(
        ax=ax_map,
        color="red",
        edgecolor="black",
        markersize=100
    )
    ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
    ax_map.set_xlim(xmin, xmax)
    ax_map.set_ylim(ymin, ymax)
    ax_map.set_title(f"PID {row['pid']} | Média: {row['mean_up_fusion']:.2f} m")
    ax_map.set_axis_off()

    # --- Série temporal original ---
    serie = up_fusion.loc[idx, colunas_temporais].astype(float)
    serie.index = datas
    ax_series.plot(serie.index, serie.values, color="blue", label="Original")
    ax_series.axhline(0, color='gray', linestyle='--', linewidth=0.7)
    ax_series.set_title("Série temporal up_fusion")
    ax_series.set_ylabel("Deslocamento (m)")
    ax_series.set_xlabel("Data")
    ax_series.grid(True)
    ax_series.legend()

    # --- Decomposição temporal ---
    # Preencher possíveis NaNs
    serie_interp = serie.interpolate(method='linear', limit_direction='both')

    # Decomposição
    decomposition = seasonal_decompose(serie_interp, model='additive', period=12)

    # Plot da tendência
    ax_trend.plot(serie.index, decomposition.trend, color='red', label='Tendência')
    ax_trend.set_title("Tendência (decompose)")
    ax_trend.set_ylabel("Deslocamento (m)")
    ax_trend.set_xlabel("Data")
    ax_trend.grid(True)
    ax_trend.legend()

plt.tight_layout()
plt.show()




In [ ]:
# Definir os limites Alqueva
norte_min = 1855050
norte_max = 1855850
este_min = 2792250
este_max = 2793250

# Aplicar o filtro
df = df[
    (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
    (df['easting'] >= este_min) & (df['easting'] <= este_max)
]

df

In [ ]:
s = df_asc.T
s.head(25)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

# Criar GeoDataFrame com EPSG:3035
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.easting, df.northing),
    crs="EPSG:3035"  # European LAEA projection
)

# Converter para Web Mercator para compatibilidade com o mapa de fundo
gdf_webmerc = gdf.to_crs(epsg=3857)

# Plot com mapa de fundo
fig, ax = plt.subplots(figsize=(10, 10))
gdf_webmerc.plot(ax=ax, color='red', markersize=50, label='Meus pontos')
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
#ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
ax.set_axis_off()
plt.legend()
plt.title("Localização dos Pontos com Mapa de Fundo")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
import geopandas as gpd

# Criar o GeoDataFrame com o CRS correto (ETRS89 / LAEA Europe)
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.easting, df.northing),
    crs="EPSG:3035"  # ou outro se souberes que não é este
)

# Converter para Web Mercator para usar no mapa
gdf_webmerc = gdf.to_crs(epsg=3857)

# Escolher a coluna para o valor
coluna_valor = "mean_velocity"

# Normalização das cores
norm = colors.TwoSlopeNorm(
    vmin=gdf[coluna_valor].min(),
    vcenter=0,
    vmax=gdf[coluna_valor].max()
)

# Colormap
cmap = plt.cm.get_cmap('jet')

# Criar figura
fig, ax = plt.subplots(figsize=(10, 10))
gdf_webmerc.plot(
    ax=ax,
    column=coluna_valor,
    cmap=cmap,
    markersize=30,
    norm=norm,
    legend=False
)

# Adicionar mapa de fundo
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()
plt.title(f"Pontos com escala de cores ({coluna_valor})")

# Adicionar barra de cores
from matplotlib.cm import ScalarMappable
sm = ScalarMappable(cmap=cmap, norm=norm)
sm._A = []  # Forçar ScalarMappable a funcionar
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label(coluna_valor)

plt.show()



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
import geopandas as gpd

# --- Criar GeoDataFrame ---
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.easting, df.northing),
    crs="EPSG:3035"
)
gdf_webmerc = gdf.to_crs(epsg=3857)

# Variáveis a representar
variaveis = [
    "height",
    "rmse",
    "mean_velocity",
    "mean_velocity_std",
    "acceleration",
    "acceleration_std",
    "seasonality",
    "seasonality_std"
]

# Criar figura 4x2 (mais altura, mais espaço)
fig, axs = plt.subplots(2, 4, figsize=(20, 10), constrained_layout=True)
axs = axs.flatten()

for i, var in enumerate(variaveis):
    ax = axs[i]

    vmin = gdf_webmerc[var].min()
    vmax = gdf_webmerc[var].max()

    # Normalização de cores
    if vmin < 0 and vmax > 0:
        norm = colors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
    else:
        norm = colors.Normalize(vmin=vmin, vmax=vmax)

    # Plot principal
    gdf_webmerc.plot(
        ax=ax,
        column=var,
        cmap='jet',
        markersize=10,
        norm=norm,
        legend=False
    )
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
    ax.set_title(var, fontsize=10)
    ax.set_axis_off()

    # Colorbar individual
    sm = plt.cm.ScalarMappable(cmap='jet', norm=norm)
    sm._A = []
    cbar = fig.colorbar(sm, ax=ax, shrink=0.6, pad=0.01)
    cbar.ax.tick_params(labelsize=7)

# Exibir
plt.show()



In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
from mpl_toolkits.axes_grid1 import make_axes_locatable

# --- Criação do GeoDataFrame com coordenadas easting/northing ---
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.easting, df.northing),
    crs="EPSG:3035"
)
gdf_webmerc = gdf.to_crs(epsg=3857)

# Nome da variável e PID de interesse
variavel_cor = "mean_velocity"
pid_escolhido = '40McjqvU5C'  # Substitui se quiseres outro

# Isolar colunas temporais
colunas_temporais = [col for col in df.columns if col.startswith("20")]
datas = pd.to_datetime(colunas_temporais)

# Verifica se o PID existe
linha_ponto = df[df["pid"] == pid_escolhido]
linha_ponto_geo = gdf_webmerc[gdf_webmerc["pid"] == pid_escolhido]

if linha_ponto.empty:
    print(f"PID {pid_escolhido} não encontrado!")
else:
    serie_temporal = linha_ponto[colunas_temporais].values.flatten().astype(float)

    # Normalização das cores
    norm = colors.TwoSlopeNorm(
        vmin=gdf_webmerc[variavel_cor].min(),
        vcenter=0,
        vmax=gdf_webmerc[variavel_cor].max()
    )

    # --- Criar figura com layout ajustado ---
    fig, axs = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [1.2, 1]})

    # --- Painel do mapa ---
    divider = make_axes_locatable(axs[0])
    cax = divider.append_axes("right", size="5%", pad=0.1)

    gdf_webmerc.plot(
        ax=axs[0],
        column=variavel_cor,
        cmap='jet',
        markersize=20,
        legend=True,
        norm=norm,
        cax=cax
    )
    ctx.add_basemap(axs[0], source=ctx.providers.Esri.WorldImagery)
    axs[0].set_title(f"Mapa com escala de {variavel_cor}")
    axs[0].set_axis_off()

    # Destacar ponto específico com círculo maior
    linha_ponto_geo.plot(
        ax=axs[0],
        color='none',
        edgecolor='black',
        linewidth=2,
        markersize=200,
        zorder=3
    )
    axs[0].scatter(
        linha_ponto_geo.geometry.x,
        linha_ponto_geo.geometry.y,
        color='yellow',
        s=80,
        zorder=4,
        edgecolor='black'
    )

    # --- Painel da série temporal ---
    axs[1].plot(datas, serie_temporal, marker='o', linestyle='-', color='blue')
    axs[1].set_title(f"Série temporal do deslocamento\nPID {pid_escolhido}")
    axs[1].set_xlabel("Data")
    axs[1].set_ylabel("Deslocamento (mm)")
    axs[1].grid(True)

    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np

# --- Criar GeoDataFrame ---
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.easting, df.northing),
    crs="EPSG:3035"
)
gdf_webmerc = gdf.to_crs(epsg=3857)

# --- Variáveis principais ---
variavel_cor = "mean_velocity"
colunas_temporais = [col for col in df.columns if col.startswith("20")]
datas = pd.to_datetime(colunas_temporais)

# --- Selecionar os 10 pontos com maior mean_velocity ---
top10 = gdf_webmerc.sort_values(by=variavel_cor, ascending=False).head(10)

# --- Normalização da cor ---
norm = colors.TwoSlopeNorm(
    vmin=gdf_webmerc[variavel_cor].min(),
    vcenter=0,
    vmax=gdf_webmerc[variavel_cor].max()
)

# --- Criar figura ---
fig, axs = plt.subplots(nrows=10, ncols=2, figsize=(14, 40), gridspec_kw={'width_ratios': [1.2, 1]})

for i, (idx, linha) in enumerate(top10.iterrows()):
    ax_map = axs[i, 0]
    ax_plot = axs[i, 1]

    # --- Mapa ---
    divider = make_axes_locatable(ax_map)
    cax = divider.append_axes("right", size="5%", pad=0.05)

    gdf_webmerc.plot(
        ax=ax_map,
        column=variavel_cor,
        cmap='jet',
        markersize=20,
        norm=norm,
        cax=cax,
        legend=True
    )
    ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
    ax_map.set_axis_off()
    ax_map.set_title(f"Mapa do PID {linha['pid']}")

    # Destacar o ponto
    ax_map.scatter(
        linha.geometry.x,
        linha.geometry.y,
        color='yellow',
        s=200,
        edgecolor='black',
        linewidth=2,
        zorder=4
    )

    # --- Série temporal ---
    serie = df.loc[df['pid'] == linha['pid'], colunas_temporais].values.flatten().astype(float)
    ax_plot.plot(datas, serie, marker='o', linestyle='-', color='blue', label='Série')

    # --- Trend line ---
    x = (datas - datas[0]).days  # dias desde o início (como base temporal)
    coeffs = np.polyfit(x, serie, deg=2)  # ajuste linear
    trend = np.poly1d(coeffs)(x)
    ax_plot.plot(datas, trend, color='red', linestyle='--', label='Trend line')

    ax_plot.set_title(f"Série temporal - PID {linha['pid']}")
    ax_plot.set_xlabel("Data")
    ax_plot.set_ylabel("Deslocamento (mm)")
    ax_plot.grid(True)
    ax_plot.legend(fontsize=8)

plt.tight_layout()
plt.show()



In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np

# --- Criar GeoDataFrame ---
gdf = gpd.GeoDataFrame(
    df_asc,  # Substitua por df_desc se necessário
    geometry=gpd.points_from_xy(df_asc.easting, df_asc.northing),
    crs="EPSG:3035"
)
gdf_webmerc = gdf.to_crs(epsg=3857)

# --- Variáveis principais ---
variavel_cor = "mean_velocity"  # Substitua pela variável desejada
colunas_temporais = [col for col in df_asc.columns if col.startswith("20")]  # Ajuste conforme necessário
datas = pd.to_datetime(colunas_temporais)

# --- Selecionar os 10 pontos com maior mean_velocity ---
top10 = gdf_webmerc.sort_values(by=variavel_cor, ascending=False).head(10)

# --- Normalização da cor ---
norm = colors.TwoSlopeNorm(
    vmin=gdf_webmerc[variavel_cor].min(),
    vcenter=0,
    vmax=gdf_webmerc[variavel_cor].max()
)

# --- Criar figura ---
fig, axs = plt.subplots(nrows=10, ncols=2, figsize=(14, 40), gridspec_kw={'width_ratios': [1.2, 1]})

for i, (idx, linha) in enumerate(top10.iterrows()):
    ax_map = axs[i, 0]
    ax_plot = axs[i, 1]

    # --- Mapa ---
    divider = make_axes_locatable(ax_map)
    cax = divider.append_axes("right", size="5%", pad=0.05)

    gdf_webmerc.plot(
        ax=ax_map,
        column=variavel_cor,
        cmap='jet',
        markersize=20,
        norm=norm,
        cax=cax,
        legend=True
    )
    ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
    ax_map.set_axis_off()
    ax_map.set_title(f"Mapa do PID {linha['pid']}")

    # Destacar o ponto
    ax_map.scatter(
        linha.geometry.x,
        linha.geometry.y,
        color='yellow',
        s=200,
        edgecolor='black',
        linewidth=2,
        zorder=4
    )

    # --- Série temporal ---
    serie = df_asc.loc[df_asc['pid'] == linha['pid'], colunas_temporais].values.flatten().astype(float)
    ax_plot.plot(datas, serie, marker='o', linestyle='-', color='blue', label='Série')

    # --- Trend line ---
    x = (datas - datas[0]).days  # dias desde o início (como base temporal)
    coeffs = np.polyfit(x, serie, deg=2)  # ajuste polinomial de grau 2
    trend = np.poly1d(coeffs)(x)
    ax_plot.plot(datas, trend, color='red', linestyle='--', label='Trend line')

    ax_plot.set_title(f"Série temporal - PID {linha['pid']}")
    ax_plot.set_xlabel("Data")
    ax_plot.set_ylabel("Deslocamento (mm)")
    ax_plot.grid(True)
    ax_plot.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
from mpl_toolkits.axes_grid1 import make_axes_locatable
from statsmodels.tsa.seasonal import seasonal_decompose
import numpy as np

# --- Criar GeoDataFrame ---
gdf = gpd.GeoDataFrame(
    df_asc,  # Substitua por df_desc se necessário
    geometry=gpd.points_from_xy(df_asc.easting, df_asc.northing),
    crs="EPSG:3035"
)
gdf_webmerc = gdf.to_crs(epsg=3857)

# --- Variáveis principais ---
variavel_cor = "mean_velocity"  # Substitua pela variável desejada
colunas_temporais = [col for col in df_asc.columns if col.startswith("20")]  # Ajuste conforme necessário
datas = pd.to_datetime(colunas_temporais)

# --- Selecionar os 10 pontos com maior mean_velocity ---
top10 = gdf_webmerc.sort_values(by=variavel_cor, ascending=False).head(10)

# --- Normalização da cor ---
norm = colors.TwoSlopeNorm(
    vmin=gdf_webmerc[variavel_cor].min(),
    vcenter=0,
    vmax=gdf_webmerc[variavel_cor].max()
)

# --- Criar figura ---
fig, axs = plt.subplots(nrows=10, ncols=2, figsize=(14, 50), gridspec_kw={'width_ratios': [1.2, 1]})

for i, (idx, linha) in enumerate(top10.iterrows()):
    ax_map = axs[i, 0]
    ax_plot = axs[i, 1]

    # --- Mapa ---
    divider = make_axes_locatable(ax_map)
    cax = divider.append_axes("right", size="5%", pad=0.05)

    gdf_webmerc.plot(
        ax=ax_map,
        column=variavel_cor,
        cmap='jet',
        markersize=20,
        norm=norm,
        cax=cax,
        legend=True
    )
    ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
    ax_map.set_axis_off()
    ax_map.set_title(f"Mapa do PID {linha['pid']}")

    # Destacar o ponto
    ax_map.scatter(
        linha.geometry.x,
        linha.geometry.y,
        color='yellow',
        s=200,
        edgecolor='black',
        linewidth=2,
        zorder=4
    )

    # --- Série temporal e decomposição ---
    serie = df_asc.loc[df_asc['pid'] == linha['pid'], colunas_temporais].values.flatten().astype(float)
    serie_indexed = pd.Series(serie, index=datas)

    # Preencher valores ausentes para decomposição
    serie_indexed = serie_indexed.interpolate(method='linear')

    # Decomposição da série temporal
    decomposition = seasonal_decompose(serie_indexed, model='additive', period=12)

    # Plotar os componentes
    ax_plot.plot(datas, decomposition.observed, label='Original', color='blue')
    ax_plot.plot(datas, decomposition.trend, label='Tendência', color='red', linestyle='--')
    ax_plot.plot(datas, decomposition.seasonal, label='Sazonalidade', color='green', linestyle='-.')
    ax_plot.plot(datas, decomposition.resid, label='Resíduos', color='orange', linestyle=':')
    
    ax_plot.set_title(f"Série temporal - PID {linha['pid']}")
    ax_plot.set_xlabel("Data")
    ax_plot.set_ylabel("Deslocamento (mm)")
    ax_plot.grid(True)
    ax_plot.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
from statsmodels.tsa.seasonal import seasonal_decompose
import numpy as np

# --- Criar GeoDataFrame ---
gdf = gpd.GeoDataFrame(
    df_asc,  # Substitua por df_desc se necessário
    geometry=gpd.points_from_xy(df_asc.easting, df_asc.northing),
    crs="EPSG:3035"
)
gdf_webmerc = gdf.to_crs(epsg=3857)

# --- Variáveis principais ---
variavel_cor = "mean_velocity"  # Substitua pela variável desejada
colunas_temporais = [col for col in df_asc.columns if col.startswith("20")]  # Ajuste conforme necessário
datas = pd.to_datetime(colunas_temporais)

# --- Selecionar os 10 pontos com maior mean_velocity ---
top10 = gdf_webmerc.sort_values(by=variavel_cor, ascending=False).head(10)

# --- Normalização da cor ---
norm = colors.TwoSlopeNorm(
    vmin=gdf_webmerc[variavel_cor].min(),
    vcenter=0,
    vmax=gdf_webmerc[variavel_cor].max()
)

# --- Criar figura ---
fig, axs = plt.subplots(nrows=10, ncols=4, figsize=(20, 50), gridspec_kw={'width_ratios': [1, 1, 1, 1]})
fig.subplots_adjust(hspace=0.5, wspace=0.3)

for i, (idx, linha) in enumerate(top10.iterrows()):
    # --- Série temporal e decomposição ---
    serie = df_asc.loc[df_asc['pid'] == linha['pid'], colunas_temporais].values.flatten().astype(float)
    serie_indexed = pd.Series(serie, index=datas)

    # Preencher valores ausentes para decomposição
    serie_indexed = serie_indexed.interpolate(method='linear')

    # Decomposição da série temporal
    decomposition = seasonal_decompose(serie_indexed, model='additive', period=12)

    # --- Gráficos separados ---
    axs[i, 0].plot(datas, decomposition.observed, label='Original', color='blue')
    axs[i, 0].set_title(f"Original - PID {linha['pid']}")
    axs[i, 0].set_xlabel("Data")
    axs[i, 0].set_ylabel("Deslocamento (mm)")
    axs[i, 0].grid(True)

    axs[i, 1].plot(datas, decomposition.trend, label='Tendência', color='red')
    axs[i, 1].set_title(f"Tendência - PID {linha['pid']}")
    axs[i, 1].set_xlabel("Data")
    axs[i, 1].set_ylabel("Deslocamento (mm)")
    axs[i, 1].grid(True)

    axs[i, 2].plot(datas, decomposition.seasonal, label='Sazonalidade', color='green')
    axs[i, 2].set_title(f"Sazonalidade - PID {linha['pid']}")
    axs[i, 2].set_xlabel("Data")
    axs[i, 2].set_ylabel("Deslocamento (mm)")
    axs[i, 2].grid(True)

    axs[i, 3].plot(datas, decomposition.resid, label='Resíduos', color='orange')
    axs[i, 3].set_title(f"Resíduos - PID {linha['pid']}")
    axs[i, 3].set_xlabel("Data")
    axs[i, 3].set_ylabel("Deslocamento (mm)")
    axs[i, 3].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
from mpl_toolkits.axes_grid1 import make_axes_locatable
from statsmodels.tsa.seasonal import seasonal_decompose
import numpy as np

# --- Criar GeoDataFrame ---
gdf = gpd.GeoDataFrame(
    df_asc,  # Substitua por df_desc se necessário
    geometry=gpd.points_from_xy(df_asc.easting, df_asc.northing),
    crs="EPSG:3035"
)
gdf_webmerc = gdf.to_crs(epsg=3857)

# --- Variáveis principais ---
variavel_cor = "mean_velocity"  # Substitua pela variável desejada
colunas_temporais = [col for col in df_asc.columns if col.startswith("20")]  # Ajuste conforme necessário
datas = pd.to_datetime(colunas_temporais)

# --- Selecionar os 10 pontos com maior mean_velocity ---
top10 = gdf_webmerc.sort_values(by=variavel_cor, ascending=False).head(10)

# --- Normalização da cor ---
norm = colors.TwoSlopeNorm(
    vmin=gdf_webmerc[variavel_cor].min(),
    vcenter=0,
    vmax=gdf_webmerc[variavel_cor].max()
)

# --- Criar figura ---
for i, (idx, linha) in enumerate(top10.iterrows()):
    fig, axs = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [1.2, 2]})
    fig.subplots_adjust(hspace=0.5, wspace=0.3)

    # --- Mapa ---
    ax_map = axs[0]
    divider = make_axes_locatable(ax_map)
    cax = divider.append_axes("right", size="5%", pad=0.05)

    gdf_webmerc.plot(
        ax=ax_map,
        column=variavel_cor,
        cmap="jet",
        markersize=20,
        norm=norm,
        legend=True,
        cax=cax
    )
    ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
    ax_map.set_title(f"Mapa - PID {linha['pid']}")
    ax_map.set_axis_off()

    # Destacar o ponto
    ax_map.scatter(
        linha.geometry.x,
        linha.geometry.y,
        color="yellow",
        s=200,
        edgecolor="black",
        linewidth=2,
        zorder=4
    )

    # --- Série temporal e decomposição ---
    serie = df_asc.loc[df_asc['pid'] == linha['pid'], colunas_temporais].values.flatten().astype(float)
    serie_indexed = pd.Series(serie, index=datas)

    # Preencher valores ausentes para decomposição
    serie_indexed = serie_indexed.interpolate(method='linear')

    # Decomposição da série temporal
    decomposition = seasonal_decompose(serie_indexed, model='additive', period=12)

    # --- Gráficos separados ---
    ax_original = fig.add_subplot(4, 2, 2)
    ax_original.plot(datas, decomposition.observed, label='Original', color='blue')
    ax_original.set_title("Série Original")
    ax_original.set_xlabel("Data")
    ax_original.set_ylabel("Deslocamento (mm)")
    ax_original.grid(True)

    ax_trend = fig.add_subplot(4, 2, 4)
    ax_trend.plot(datas, decomposition.trend, label='Tendência', color='red')
    ax_trend.set_title("Tendência")
    ax_trend.set_xlabel("Data")
    ax_trend.set_ylabel("Deslocamento (mm)")
    ax_trend.grid(True)

    ax_seasonal = fig.add_subplot(4, 2, 6)
    ax_seasonal.plot(datas, decomposition.seasonal, label='Sazonalidade', color='green')
    ax_seasonal.set_title("Sazonalidade")
    ax_seasonal.set_xlabel("Data")
    ax_seasonal.set_ylabel("Deslocamento (mm)")
    ax_seasonal.grid(True)

    ax_resid = fig.add_subplot(4, 2, 8)
    ax_resid.plot(datas, decomposition.resid, label='Resíduos', color='orange')
    ax_resid.set_title("Resíduos")
    ax_resid.set_xlabel("Data")
    ax_resid.set_ylabel("Deslocamento (mm)")
    ax_resid.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np

# --- Criar GeoDataFrame ---
gdf = gpd.GeoDataFrame(
    df_asc,  # Substitua por df_desc se necessário
    geometry=gpd.points_from_xy(df_asc.easting, df_asc.northing),
    crs="EPSG:3035"
)
gdf_webmerc = gdf.to_crs(epsg=3857)

# --- Variáveis principais ---
variavel_cor = "mean_velocity"  # Substitua pela variável desejada
colunas_temporais = [col for col in df_asc.columns if col.startswith("20")]  # Ajuste conforme necessário
datas = pd.to_datetime(colunas_temporais)

# --- Selecionar os 10 pontos com maior mean_velocity ---
top10 = gdf_webmerc.sort_values(by=variavel_cor, ascending=False).head(10)

# --- Normalização da cor ---
norm = colors.TwoSlopeNorm(
    vmin=gdf_webmerc[variavel_cor].min(),
    vcenter=0,
    vmax=gdf_webmerc[variavel_cor].max()
)

# --- Criar figura ---
for i, (idx, linha) in enumerate(top10.iterrows()):
    fig, axs = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [1.2, 2]})
    fig.subplots_adjust(hspace=0.5, wspace=0.3)

    # --- Mapa ---
    ax_map = axs[0]
    divider = make_axes_locatable(ax_map)
    cax = divider.append_axes("right", size="5%", pad=0.05)

    gdf_webmerc.plot(
        ax=ax_map,
        column=variavel_cor,
        cmap="jet",
        markersize=20,
        norm=norm,
        legend=True,
        cax=cax
    )
    ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
    ax_map.set_title(f"Mapa - PID {linha['pid']}")
    ax_map.set_axis_off()

    # Destacar o ponto
    ax_map.scatter(
        linha.geometry.x,
        linha.geometry.y,
        color="yellow",
        s=200,
        edgecolor="black",
        linewidth=2,
        zorder=4
    )

    # --- Série temporal e análise de Fourier ---
    serie = df_asc.loc[df_asc['pid'] == linha['pid'], colunas_temporais].values.flatten().astype(float)
    serie_indexed = pd.Series(serie, index=datas)

    # Preencher valores ausentes
    serie_indexed = serie_indexed.interpolate(method='linear')

    # Aplicar Transformada de Fourier
    fft = np.fft.fft(serie_indexed)
    freqs = np.fft.fftfreq(len(fft), d=1)  # Frequências (assumindo espaçamento uniforme)

    # Magnitude do espectro
    magnitude = np.abs(fft)

    # --- Gráficos separados ---
    ax_original = fig.add_subplot(4, 2, 2)
    ax_original.plot(datas, serie_indexed, label='Original', color='blue')
    ax_original.set_title("Série Original")
    ax_original.set_xlabel("Data")
    ax_original.set_ylabel("Deslocamento (mm)")
    ax_original.grid(True)

    ax_fft = fig.add_subplot(4, 2, 4)
    ax_fft.plot(freqs[:len(freqs)//2], magnitude[:len(magnitude)//2], label='Espectro de Fourier', color='red')
    ax_fft.set_title("Espectro de Fourier")
    ax_fft.set_xlabel("Frequência")
    ax_fft.set_ylabel("Magnitude")
    ax_fft.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Exemplo de série temporal (substitua pela sua série)
datas = pd.date_range(start="2020-01-01", periods=100, freq="D")
serie_temporal = np.sin(2 * np.pi * 0.1 * np.arange(100)) + np.random.normal(0, 0.1, 100)

# Aplicar a Transformada de Fourier
fft = np.fft.fft(serie_temporal)
freqs = np.fft.fftfreq(len(fft), d=1)  # d=1 assume espaçamento uniforme de 1 unidade de tempo

# Calcular a magnitude
magnitude = np.abs(fft)

# Filtrar apenas as frequências positivas
freqs_positivas = freqs[freqs > 0]
magnitude_positiva = magnitude[freqs > 0]

# Plotar o espectro de frequência
plt.figure(figsize=(10, 6))
plt.plot(freqs_positivas, magnitude_positiva, label="Espectro de Fourier", color="blue")
plt.title("Espectro de Frequência")
plt.xlabel("Frequência (Hz)")
plt.ylabel("Magnitude")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Exemplo de série temporal (substitua pela sua série)
datas = pd.date_range(start="2020-01-01", periods=100, freq="6D")
serie_temporal = np.sin(2 * np.pi * 0.1 * np.arange(100)) + np.random.normal(0, 0.1, 100)

# Aplicar a Transformada de Fourier
fft = np.fft.fft(serie_temporal)
freqs = np.fft.fftfreq(len(fft), d=6)  # d=6 porque os dados são de 6 em 6 dias

# Calcular a magnitude
magnitude = np.abs(fft)

# Filtrar apenas as frequências positivas
freqs_positivas = freqs[freqs > 0]
magnitude_positiva = magnitude[freqs > 0]

# Plotar o espectro de frequência
plt.figure(figsize=(10, 6))
plt.plot(freqs_positivas, magnitude_positiva, label="Espectro de Fourier", color="blue")
plt.title("Espectro de Frequência")
plt.xlabel("Frequência (1/dias)")
plt.ylabel("Magnitude")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
from mpl_toolkits.axes_grid1 import make_axes_locatable

# --- Criação do GeoDataFrame com coordenadas easting/northing ---
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.easting, df.northing),
    crs="EPSG:3035"  # CRS dos dados fornecidos
)
gdf_webmerc = gdf.to_crs(epsg=3857)  # Para compatibilidade com o basemap

# Nome da variável e limite
variavel_cor = "mean_velocity"
limite = 0.5

# Isolar colunas com datas (começam com "20")
colunas_temporais = [col for col in df.columns if col.startswith("20")]
datas = pd.to_datetime(colunas_temporais)

# --- Filtrar os pontos para destacar ---
pontos_destacados = df[df[variavel_cor] > limite]
pontos_destaque = gdf_webmerc[gdf_webmerc[variavel_cor] > limite]

# Normalização das cores
norm = colors.TwoSlopeNorm(
    vmin=gdf_webmerc[variavel_cor].min(),
    vcenter=0,
    vmax=gdf_webmerc[variavel_cor].max()
)

# --- Criar figura com layout ajustado ---
fig, axs = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [1.2, 1]})

# --- Painel do mapa ---
divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.1)

gdf_webmerc.plot(
    ax=axs[0],
    column=variavel_cor,
    cmap='jet',
    markersize=20,
    legend=True,
    norm=norm,
    cax=cax
)
ctx.add_basemap(axs[0], source=ctx.providers.Esri.WorldImagery)
axs[0].set_title(f"Mapa com escala de {variavel_cor}")
axs[0].set_axis_off()

# --- Destacar pontos acima do limite ---
pontos_destaque.plot(
    ax=axs[0],
    facecolor='red',
    edgecolor='black',
    markersize=80,
    linewidth=1,
    label=f"{variavel_cor} > {limite} mm/ano"
)

# --- Painel da série temporal ---
for idx, row in pontos_destacados.iterrows():
    serie = row[colunas_temporais].values.astype(float)
    axs[1].plot(datas, serie, alpha=0.5)

axs[1].set_title(f"Séries temporais dos pontos com {variavel_cor} > {limite} mm/ano")
axs[1].set_xlabel("Data")
axs[1].set_ylabel("Deslocamento (mm)")
axs[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
from mpl_toolkits.axes_grid1 import make_axes_locatable

# --- Criação do GeoDataFrame com coordenadas easting/northing ---
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.easting, df.northing),
    crs="EPSG:3035"  # CRS dos dados fornecidos
)
gdf_webmerc = gdf.to_crs(epsg=3857)  # Para compatibilidade com o basemap

# Nome da variável e limite
variavel_cor = "mean_velocity"
limite = -0.5

# Isolar colunas com datas (começam com "20")
colunas_temporais = [col for col in df.columns if col.startswith("20")]
datas = pd.to_datetime(colunas_temporais)

# --- Filtrar os pontos para destacar ---
pontos_destacados = df[df[variavel_cor] > limite]
pontos_destaque = gdf_webmerc[gdf_webmerc[variavel_cor] > limite]

# Normalização das cores
norm = colors.TwoSlopeNorm(
    vmin=gdf_webmerc[variavel_cor].min(),
    vcenter=0,
    vmax=gdf_webmerc[variavel_cor].max()
)

# --- Criar figura com layout ajustado ---
fig, axs = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [1.2, 1]})

# --- Painel do mapa ---
divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.1)

gdf_webmerc.plot(
    ax=axs[0],
    column=variavel_cor,
    cmap='jet',
    markersize=20,
    legend=True,
    norm=norm,
    cax=cax
)
ctx.add_basemap(axs[0], source=ctx.providers.Esri.WorldImagery)
axs[0].set_title(f"Mapa com escala de {variavel_cor}")
axs[0].set_axis_off()

# --- Destacar todos os pontos acima do limite com círculos maiores ---
for idx, ponto in pontos_destaque.iterrows():
    x, y = ponto.geometry.x, ponto.geometry.y
    axs[0].scatter(x, y, color='yellow', s=400, edgecolor='black', linewidth=2)

# --- Destacar o ponto original (ponto com marcador menor) ---
pontos_destaque.plot(
    ax=axs[0],
    facecolor='red',
    edgecolor='black',
    markersize=80,
    linewidth=1,
    label=f"{variavel_cor} > {limite} mm/ano"
)

# --- Painel da série temporal ---
for idx, row in pontos_destacados.iterrows():
    serie = row[colunas_temporais].values.astype(float)
    axs[1].plot(datas, serie, alpha=0.5)

axs[1].set_title(f"Séries temporais dos pontos com {variavel_cor} > {limite} mm/ano")
axs[1].set_xlabel("Data")
axs[1].set_ylabel("Deslocamento (mm)")
axs[1].grid(True)

# Ajuste no layout para garantir que tudo caiba bem
plt.tight_layout()
plt.show()



In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
from mpl_toolkits.axes_grid1 import make_axes_locatable

# --- Criação do GeoDataFrame com coordenadas easting/northing ---
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.easting, df.northing),
    crs="EPSG:3035"
)
gdf_webmerc = gdf.to_crs(epsg=3857)

# Nome da variável e limite
variavel_cor = "mean_velocity"
limite = -0.5

# Colunas temporais (as que começam por "20")
colunas_temporais = [col for col in df.columns if col.startswith("20")]
datas = pd.to_datetime(colunas_temporais)

# Filtrar os pontos a destacar
pontos_destacados = df[df[variavel_cor] > limite]
pontos_destaque = gdf_webmerc[gdf_webmerc[variavel_cor] > limite]

# Normalização da cor
norm = colors.TwoSlopeNorm(
    vmin=gdf_webmerc[variavel_cor].min(),
    vcenter=0,
    vmax=gdf_webmerc[variavel_cor].max()
)

# Criar figura
fig, axs = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [1.2, 1]})

# --- Painel do mapa ---
divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.1)

gdf_webmerc.plot(
    ax=axs[0],
    column=variavel_cor,
    cmap='jet',
    markersize=20,
    legend=True,
    norm=norm,
    cax=cax
)
ctx.add_basemap(axs[0], source=ctx.providers.Esri.WorldImagery)
axs[0].set_title(f"Mapa com escala de {variavel_cor}")
axs[0].set_axis_off()

# --- Destacar os pontos com círculos e legendas com PID ---
for idx, ponto in pontos_destaque.iterrows():
    x, y = ponto.geometry.x, ponto.geometry.y
    axs[0].scatter(x, y, color='yellow', s=400, edgecolor='black', linewidth=2)
    axs[0].annotate(
        texto := ponto["pid"],
        (x, y),
        textcoords="offset points",
        xytext=(5, 5),
        fontsize=7,
        color='white',
        weight='bold',
        bbox=dict(boxstyle="round,pad=0.2", fc="black", alpha=0.6)
    )

# --- Reforçar os pontos destacados em vermelho ---
pontos_destaque.plot(
    ax=axs[0],
    facecolor='red',
    edgecolor='black',
    markersize=80,
    linewidth=1,
    label=f"{variavel_cor} > {limite} mm/ano"
)

# --- Painel da série temporal ---
for idx, row in pontos_destacados.iterrows():
    serie = row[colunas_temporais].values.astype(float)
    pid = row["pid"]
    axs[1].plot(datas, serie, alpha=0.6, label=pid)  # Adiciona label com o PID

axs[1].set_title(f"Séries temporais dos pontos com {variavel_cor} > {limite} mm/ano")
axs[1].set_xlabel("Data")
axs[1].set_ylabel("Deslocamento (mm)")
axs[1].grid(True)

# Mostrar legenda (com número limitado para não sobrecarregar)
axs[1].legend(loc="best", fontsize=8, ncol=2, title="PID")

plt.tight_layout()
plt.show()


In [ ]:
import folium
import pandas as pd

# Dados dos teus pontos
#df = pd.read_csv("data/meus_pontos.csv")

# Centro do mapa
mapa = folium.Map(location=[df['easting'].mean(), df['northing'].mean()], zoom_start=12)

# Adiciona cada ponto
for _, row in df.iterrows():
    folium.Marker(
        location=[row['easting'], row['northing']],
        popup=f"Ponto: {row['nome'] if 'nome' in row else ''}"
    ).add_to(mapa)

# Mostrar (num Jupyter Notebook aparece diretamente)
mapa.save("meu_mapa.html")

In [ ]:
import folium
from folium.plugins import MarkerCluster

# Converter para lat/lon para o mapa (Folium exige EPSG:4326)
gdf_latlon = gdf.to_crs(epsg=4326)

# Criar mapa centrado nos dados
centro = [gdf_latlon.geometry.y.mean(), gdf_latlon.geometry.x.mean()]
m = folium.Map(location=centro, zoom_start=10, tiles='Esri.WorldImagery')

# Cluster para os pontos
marker_cluster = MarkerCluster().add_to(m)

# Colormap simples (poderíamos fazer algo mais customizado com branca ou branca-colormap)
for idx, row in gdf_latlon.iterrows():
    valor = row[variavel_cor]
    cor = 'red' if valor > limite else 'blue'
    popup_text = f"PID: {row['pid']}<br>{variavel_cor}: {valor:.2f} mm/ano"
    
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=6,
        color='black',
        fill=True,
        fill_color=cor,
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=200)
    ).add_to(marker_cluster)

# Mostrar
m.save("mapa_folium.html")
